In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModel, pipeline
from transformers.pipelines.pt_utils import KeyDataset
from tqdm import tqdm

from torch.utils.data import TensorDataset, DataLoader

from mhcicleave.lightning_model import CleavageLightningModule
from mhcicleave.models import MLP
from mhcicleave.cleavagedatasets import MultiFastaWindowDataset

import lightning.pytorch as pl

class Embedder:

    def __init__(self, name: str = "facebook/esm2_t33_650M_UR50D", device='cpu'):

        ALLOWED_MODELS = ['Rostlab/prot_bert', 'Rostlab/prot_t5_xl_uniref50', 'facebook/esm2_t33_650M_UR50D', 'onehot']
        assert name in ALLOWED_MODELS, f"This function is only built for {ALLOWED_MODELS}, get {name}"

        self.device = device
        print('Initializing protein sequence embedder ... ', end='')
        # logging.info(f"Loading ESM-2 model ({name})...")

        if name == 'Rostlab/prot_bert':
            self.featureLen = 1024
            self.tokenizer = BertTokenizer.from_pretrained(name, do_lower_case=False)
            self.model = BertModel.from_pretrained(name).to(self.device).eval()
            self.encode = self.encode_prot

        elif name == 'Rostlab/prot_t5_xl_uniref50':
            self.featureLen = 1024
            self.tokenizer = T5Tokenizer.from_pretrained(name, do_lower_case=False)
            self.model = T5EncoderModel.from_pretrained(name).to(self.device).eval()
            self.encode = self.encode_prot

        elif name == 'facebook/esm2_t33_650M_UR50D':
            self.featureLen = 1280
            self.tokenizer = AutoTokenizer.from_pretrained(name)
            self.model = AutoModel.from_pretrained(name).to(self.device).eval()
            self.encode = self.encode_esm

        elif name == 'onehot':
            self.featureLen = 21
            self.encode = self.encode_onehot

    def encode_esm(self, sequences, emb_type='concat'):

        pipe = pipeline('feature-extraction', model=self.model, tokenizer=self.tokenizer, framework='pt')
        key_dataset = KeyDataset(sequences, 'window')

        embeddings_list = []
        for out in tqdm(pipe(key_dataset, batch_size=17, return_tensors=True, num_workers=0, shuffle=False),
                        total=len(sequences), desc='Generating Embeddings'):
            embeddings = out[:, 1:-1, :]
            if emb_type == 'concat':
                emb = torch.flatten(embeddings, start_dim=1)
            embeddings_list.append(emb)
        embeddings_result = torch.stack(embeddings_list)
        # del self.model

        return embeddings_result

2025-10-16 14:07:05.725233: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-16 14:07:06.041996: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760634426.163775   10179 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760634426.199444   10179 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760634426.472433   10179 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [3]:
# args
device = 'cuda'
fasta_file = "vax_seqs_test.fasta"
checkpoint = "../models/best_model-epoch=13-val_loss=0.32.ckpt"

In [4]:
embedder = Embedder('facebook/esm2_t33_650M_UR50D', device)

Initializing protein sequence embedder ... 

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
seq_embeddings = embedder.encode_esm(dataset)

In [3]:
%%time

#torch.set_float32_matmul_precision('medium' | 'high')
torch.set_float32_matmul_precision('medium')
device = torch.device(device if torch.cuda.is_available() else 'cpu')


# Modelo MLP:
seq_len = 10 # largo cleavage window
esm2_dim = 1280
input_size_mlp = seq_len * esm2_dim

model_params_mlp = {"input_size": input_size_mlp, "hidden_linear1": 256, "hidden_linear2": 128, "out_neurons": 1, "dropout": 0.5}
model_network = MLP(**model_params_mlp)


print(f"Loading cleavage model {checkpoint} ...")
try:
    model = CleavageLightningModule.load_from_checkpoint("../models/best_model-epoch=13-val_loss=0.32.ckpt",model=model_network)
except Exception as e:
    print(f"Error loading model. Make sure the class 'GenericClassifier' is defined and imported correctly.")
    raise e

trainer = pl.Trainer()
embedder = Embedder('facebook/esm2_t33_650M_UR50D', device)

prot_results = []

###
UPSTREAM = 6
DOWNSTREAM = 4
dataset = MultiFastaWindowDataset(
    fasta_file="vax_sequences_multi_mhc1m2.fasta",
    upstream=UPSTREAM,
    downstream=DOWNSTREAM
)

seq_embeddings = embedder.encode_esm(dataset)

dataset = TensorDataset(seq_embeddings)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=10)

trainer = pl.Trainer()
predictions = trainer.predict(model, dataloader)
predictions = torch.cat([batch.flatten() for batch in predictions]).numpy()

#pd_out = format_prediction(predictions)
#prot_results.append(pd_out)

Loading cleavage model ../models/best_model-epoch=13-val_loss=0.32.ckpt ...


/home/vilion/miniforge3/envs/pLM/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: FutureWarning: The default value for `maximize` will be changed from `True` to `None` in v1.7.0 of TorchMetrics,will automatically infer the value based on the `higher_is_better` attribute of the metric (if such attribute exists) or raise an error if it does not. If you are explicitly setting the `maximize` argument to either `True` or `False` already, you can ignore this warning.
  warnings.warn(*args, **kwargs)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Initializing protein sequence embedder ... 

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
Generating Embeddings: 100%|███████████████████████████████████████████████████████████████████████████████████| 18000/18000 [00:27<00:00, 652.26it/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |                                                                                                 …

CPU times: user 26.4 s, sys: 5.98 s, total: 32.4 s
Wall time: 35.4 s


In [10]:
window_dataset = MultiFastaWindowDataset(
    fasta_file="vax_sequences_multi_mhc1m2.fasta",
    upstream=UPSTREAM,
    downstream=DOWNSTREAM
)
pd.DataFrame(list(window_dataset))

,prot_id,position,window
0,seq1_QAHG,1,XXXXXCALNN
1,seq1_QAHG,2,XXXXCALNNQ
2,seq1_QAHG,3,XXXCALNNQA
3,seq1_QAHG,4,XXCALNNQAH
4,seq1_QAHG,5,XCALNNQAHG
...,...,...,...
17995,seq1000_PASM,14,MDPLVVAASI
17996,seq1000_PASM,15,DPLVVAASIX
17997,seq1000_PASM,16,PLVVAASIXX
17998,seq1000_PASM,17,LVVAASIXXX


In [4]:
predictions

array([0.1461888 , 0.06344055, 0.40220496, ..., 0.38582447, 0.01349459,
       0.30409703], dtype=float32)

In [9]:
#1min/10min/60min
36000*10,360000*6

(360000, 2160000)

In [3]:
5000*2*31

310000

In [5]:
predictions.shape

(18000,)

In [9]:
dataset.tensors[0].shape

torch.Size([18000, 1, 12800])